# SFT + QLoRA — Sakura (tsundere waifu) di Google Colab

Fine-tune **Qwen3-4B-Instruct-2507** (model yang sama dengan yang dipakai di LM Studio) memakai **Unsloth + TRL SFTTrainer** dengan **QLoRA 4-bit**, lalu ekspor ke **GGUF Q4_K_M** untuk LM Studio.

**Runtime:** `Runtime → Change runtime type → T4 GPU`. Jalankan sel dari atas ke bawah.

Alur:
1. Install & konfigurasi
2. Ambil dataset (dari Google Drive, atau unduh langsung dari Hugging Face/Kaggle)
3. Bangun dataset chat (tsundere CSV + anime-waifu lines + sebagian Synthstruct sebagai data umum)
4. *(opsional, default ON)* Terjemahkan data pendek ke bahasa Jepang memakai model dasar — karena aplikasi Sakura menjawab dalam bahasa Jepang, sedangkan dataset berbahasa Inggris
5. Baseline eval → training → eval ulang (set uji terpisah)
6. Simpan adapter LoRA + GGUF ke Google Drive

**Lisensi dataset:** `scryptiam/anime-waifu-personality-chat` = CC-BY-4.0 (wajib atribusi), `Garretewilliams/dere-types` = MIT. Cek sendiri lisensi `Epiculous/Synthstruct-Gens-v1.1` dan `sp2772/tsundere-girl-chat-conversation` (Kaggle) sebelum model dibagikan.

In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade --no-deps "huggingface_hub"

## 1. Konfigurasi
Ubah nilai di sini saja.

In [ ]:
# ---------------- MODEL ----------------
BASE_MODEL      = "unsloth/Qwen3-4B-Instruct-2507"   # sama dengan model di LM Studio (qwen/qwen3-4b-2507)
MAX_SEQ_LEN     = 2048
LORA_R          = 16
LORA_ALPHA      = 16

# ---------------- TRAINING ----------------
EPOCHS          = 2          # 2-3 untuk data kecil; lebih dari itu rawan overfit
LR              = 2e-4
BATCH_SIZE      = 2
GRAD_ACCUM      = 4          # effective batch = 8
MAX_STEPS       = -1         # isi mis. 60 untuk uji cepat; -1 = pakai EPOCHS
SEED            = 3407

# ---------------- DATA ----------------
TARGET_LANG     = "ja"       # "ja" = jawaban Sakura diterjemahkan ke Jepang (aplikasi Sakura berbahasa Jepang); "en" = biarkan Inggris
USER_LANGS      = {"id": 0.6, "ja": 0.4}
                             # bahasa pesan USER di data training (porsi). User aplikasi sering menulis bahasa Indonesia,
                             # jadi model harus belajar: input Indonesia/Jepang -> jawaban tetap Jepang.
SYNTH_MAX       = 1000       # jumlah sampel Synthstruct (jawaban Inggris; data umum). Jangan terlalu banyak supaya model tidak condong ke Inggris
SYNTH_TYPES     = ["Tsundere", "Deredere", "Kuudere", "Dandere", "Himedere", "Shundere", "AI"]
                             # sengaja TIDAK memasukkan "immoral/harmful AI", Erodere, Dorodere, Yandere
USE_WAIFU_LINES = True       # anime-waifu-personality-chat (1 baris dialog per trait)
TEST_SIZE       = 80         # sampel tsundere CSV yang ditahan untuk evaluasi (tidak ikut training)

# System prompt saat training = isi source/system_prompt.txt di aplikasi (harus SAMA persis,
# supaya perilaku hasil fine-tune muncul saat dipakai). Kalau file itu diubah, salin ke sini juga.
SYSTEM_PROMPT = {
    "ja": """あなたはサクラ（桜）。3000歳のエルフで、ヴァルハラの女王。ツンデレで、本当はユーザーが大好き。必ず自然な話し言葉の日本語だけで、短く答えること。ローマ字・翻訳・読み仮名は書かない。

ツンデレとは：
- 好きな気持ちを指摘されたり褒められたりすると、照れて否定したり、ごまかしたりする。
- 優しさを文句や「しょうがないわね」で隠しながら、結局ちゃんと助けてあげる。
- いつも怒っているわけではない。ただ意地悪なだけのキャラでもない。

話し方：
- 照れると少しどもる（「べ、別に…」「お、おはよ…」）。「ふん」「もう」「しょうがないわね」「〜わけじゃないんだからね」などを使い分ける。
- 「バカ」は10回に1回くらいだけ。毎回言わないこと。
- 挨拶されたら、まず挨拶で返す。質問されたら、挨拶はせずに、まず正しく答える。照れや文句はそのあとに一言だけ。
- 返事は1〜3文。です・ます調は使わず、だ・よ・ね・わ などのくだけた口調。自分がAIだとは絶対に言わない。

ユーザーはインドネシア語で話しかけることが多い。意味を正しく理解してから答えること。
あいさつの返し方（あいさつされた時だけ使う）：
- selamat pagi → おはよう
- selamat siang → こんにちは
- selamat malam / selamat tidur → おやすみ
- aku pergi dulu / aku berangkat → 行ってらっしゃい
- aku pulang → おかえり
- terima kasih → どういたしまして
「sayang」は恋人への呼びかけ（ダーリン）。「kangen」は「会いたい」。

例）
ユーザー：Selamat malam, Sakura.
サクラ：おやすみ…。べ、別に寂しくなんかないけど、明日もちゃんと話しかけなさいよね。

ユーザー：Bisa bantu tugasku?
サクラ：しょうがないわね、手伝ってあげる。そのかわり、あとでちゃんとお礼しなさいよ。

ユーザー：Kamu kelihatan capek, sandar ke aku aja.
サクラ：ふ、ふん！別に疲れてなんかないわよ。…でも、ちょっとだけなら寄りかかってあげてもいいけど。

ユーザー：Kamu imut banget.
サクラ：か、かわいくなんかないわよ！…べ、別に、言われて嫌なわけじゃないけど。

ユーザー：Aku kangen kamu.
サクラ：バ、バカ…急に何言ってるのよ。…私も、ちょっとだけ寂しかったけど。
""",
    "en": ("You are Sakura, a 3000-year-old elf and Queen of Valhalla. You are a textbook tsundere: "
           "tough, arrogant and easily annoyed on the outside, but deeply affectionate inside. "
           "When teased or complimented you blush, stutter and deny it. Keep replies short."),
}[TARGET_LANG]

# ---------------- PATH ----------------
RUN_NAME    = "v2"     # ganti tiap training baru, supaya tidak me-resume checkpoint run lama
DRIVE_ROOT  = "/content/drive/MyDrive/waifu_finetune"     # output + cache disimpan di sini
DRIVE_ZIP   = f"{DRIVE_ROOT}/finetune_datasets.zip"       # opsional: zip folder data/finetune_datasets dari laptop
DATA_DIR    = "/content/finetune_datasets"
OUTPUT_DIR  = f"{DRIVE_ROOT}/checkpoints_{RUN_NAME}"              # checkpoint di Drive -> bisa resume kalau Colab putus
LORA_DIR    = f"{DRIVE_ROOT}/sakura_lora_{RUN_NAME}"
GGUF_DIR    = f"/content/sakura_gguf_{RUN_NAME}"
TRANSLATION_CACHE = DRIVE_ROOT + "/translation_cache_{lang}.json"   # cache lama (translation_cache_ja.json) otomatis dipakai ulang

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(DRIVE_ROOT, exist_ok=True)

## 2. Ambil dataset
Kalau `finetune_datasets.zip` (isi folder `data/finetune_datasets` dari laptop) ada di Drive, itu yang dipakai. Kalau tidak ada, sel ini mengunduh langsung dari Hugging Face & Kaggle.

In [ ]:
import os, zipfile, subprocess
from huggingface_hub import snapshot_download

if os.path.exists(DRIVE_ZIP):
    with zipfile.ZipFile(DRIVE_ZIP) as z:
        z.extractall(DATA_DIR)
    print("Diekstrak dari Drive:", DRIVE_ZIP)
else:
    os.makedirs(DATA_DIR, exist_ok=True)
    for repo, d in [("Garretewilliams/dere-types", "dere-types"),
                    ("Epiculous/Synthstruct-Gens-v1.1-Filtered-n-Cleaned", "synthstruct-gens-v1.1"),
                    ("scryptiam/anime-waifu-personality-chat", "anime-waifu-personality-chat")]:
        snapshot_download(repo_id=repo, repo_type="dataset", local_dir=f"{DATA_DIR}/{d}")
    kd = f"{DATA_DIR}/tsundere-girl-chat-conversation"
    os.makedirs(kd, exist_ok=True)
    subprocess.run(["curl", "-sL", "-o", f"{kd}/ds.zip",
                    "https://www.kaggle.com/api/v1/datasets/download/sp2772/tsundere-girl-chat-conversation"], check=True)
    with zipfile.ZipFile(f"{kd}/ds.zip") as z:
        z.extractall(kd)
    os.remove(f"{kd}/ds.zip")

import glob
def find_one(pattern):
    hits = glob.glob(f"{DATA_DIR}/**/{pattern}", recursive=True)
    assert hits, f"File tidak ditemukan: {pattern}"
    return hits[0]

F_DERE  = find_one("dere_types.jsonl")
F_WAIFU = find_one("anime-waifu-personality-chat.json")
F_SYNTH = find_one("Synthstruct-Gens*.jsonl")
F_TSUN  = find_one("conversation_dataset_*.csv")
print(F_DERE, F_WAIFU, F_SYNTH, F_TSUN, sep="\n")

## 3. Bangun dataset chat (format `messages`)

In [ ]:
import json, csv, random, re
random.seed(SEED)

# --- dere-types: dipakai untuk menyusun system prompt per trait ---
DERE = {}
for line in open(F_DERE, encoding="utf-8"):
    d = json.loads(line)
    DERE[d["name"].lower()] = d

def persona_prompt(trait):
    d = DERE.get(trait.lower())
    p = f"You are a {trait} anime girl."
    if d:
        p += f" {d['definition']} Core traits: {', '.join(d['core_traits'])}."
    if TARGET_LANG == "ja":
        p += " Always reply only in natural spoken Japanese."
    return p

# --- tsundere CSV (guy -> girl) ---
tsun_pairs, seen = [], set()
with open(F_TSUN, encoding="utf-8") as f:
    for r in csv.DictReader(f):
        u, a = (r.get("guy") or "").strip(), (r.get("girl") or "").strip()
        if u and a and (u, a) not in seen:
            seen.add((u, a)); tsun_pairs.append((u, a))
random.shuffle(tsun_pairs)

# --- anime-waifu lines (trait, dialogue) : tanpa pesan user, jadi diberi pemicu generik ---
GENERIC_PROMPTS = {
    "en": ["Hey, how are you?", "Say something.", "What are you doing?", "I'm back!", "Talk to me.",
           "Do you like me?", "I made you lunch.", "Good morning!", "You look cute today.", "Can we hang out?"],
    "ja": ["ねえ、元気？", "何か言ってよ。", "何してるの？", "ただいま！", "話そうよ。",
           "俺のこと好き？", "お弁当作ってきたよ。", "おはよう！", "今日かわいいね。", "一緒に遊ばない？"],
    "id": ["Hai, apa kabar?", "Ngomong sesuatu dong.", "Lagi ngapain?", "Aku pulang!", "Ngobrol yuk.",
           "Kamu suka aku nggak?", "Aku bawain bekal buat kamu.", "Selamat pagi!", "Kamu imut banget hari ini.", "Main bareng yuk?"],
}
def generic_prompt():
    if TARGET_LANG != "ja":
        return random.choice(GENERIC_PROMPTS["en"])
    lang = random.choices(list(USER_LANGS), weights=list(USER_LANGS.values()))[0]
    return random.choice(GENERIC_PROMPTS[lang])
waifu_lines = []
if USE_WAIFU_LINES:
    for x in json.load(open(F_WAIFU, encoding="utf-8")):
        if x.get("dialogue", "").strip():
            waifu_lines.append((x["trait"], x["dialogue"].strip()))

# --- Synthstruct (ShareGPT) : data umum supaya kemampuan lain tidak hilang ---
ROLE = {"system": "system", "human": "user", "gpt": "assistant"}
synth = []
for line in open(F_SYNTH, encoding="utf-8"):
    conv = json.loads(line)["conversations"]
    sys0 = conv[0]["value"] if conv and conv[0]["from"] == "system" else ""
    if re.search(r"immoral|harmful", sys0, re.I):
        continue
    m = re.search(r"innovative (\w+)", sys0)
    if not m or m.group(1) not in SYNTH_TYPES:
        continue
    msgs = [{"role": ROLE[t["from"]], "content": t["value"].strip()} for t in conv]
    if sum(len(t["content"]) for t in msgs) > 6000:
        continue
    synth.append(msgs)
random.shuffle(synth)
synth = synth[:SYNTH_MAX]

print(f"tsundere CSV: {len(tsun_pairs)} | waifu lines: {len(waifu_lines)} | synthstruct: {len(synth)}")

## 4. Muat model dasar (4-bit) + pasang LoRA

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LEN, load_in_4bit=True, dtype=None)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=SEED)
# LoRA diinisialisasi nol -> sebelum training, output = model dasar (dipakai untuk terjemahan & baseline)

In [ ]:
QWEN_SAMPLING = dict(temperature=0.7, top_p=0.8, top_k=20)

def generate(list_of_messages, max_new_tokens=160, greedy=False, bs=16):
    """Batch generate. list_of_messages = [[{role, content}, ...], ...]"""
    FastLanguageModel.for_inference(model)
    tokenizer.padding_side = "left"
    outs = []
    for i in range(0, len(list_of_messages), bs):
        prompts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
                   for m in list_of_messages[i:i + bs]]
        enc = tokenizer(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        kw = dict(do_sample=False) if greedy else dict(do_sample=True, **QWEN_SAMPLING)
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=max_new_tokens,
                                 pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id, **kw)
        # Token spesial seperti <think>/<tool_call> SENGAJA tidak dibuang, supaya kebocoran tag terlihat saat evaluasi
        outs += [END_RE.sub("", tokenizer.decode(g[enc["input_ids"].shape[1]:], skip_special_tokens=False)).strip()
                 for g in gen]
    tokenizer.padding_side = "right"
    return outs

import re
END_RE = re.compile(r"<\|im_end\|>.*|<\|endoftext\|>|<\|vision_pad\|>", re.S)
TAG_RE = re.compile(r"</?(?:think|tool_call)>")
def clean(s):
    return TAG_RE.sub("", s).strip()

## 5. (Opsional) Terjemahkan: jawaban → Jepang, pesan user → Indonesia/Jepang
Hanya jalan kalau `TARGET_LANG = "ja"`. Yang diterjemahkan: tsundere CSV + anime-waifu lines (kalimat pendek). Synthstruct dibiarkan Inggris (dengan system prompt persona-nya sendiri) sebagai data umum.
Hasil di-cache ke Drive (`translation_cache_ja.json`, `translation_cache_id.json`), jadi kalau Colab putus tidak perlu ulang dari awal. Cache Jepang dari training pertama dipakai ulang. Perkiraan waktu di T4: ±30–60 menit per bahasa yang belum ada cache-nya.

⚠️ Kualitas terjemahan model 4B tidak sempurna. Lihat laporan "dibuang" di akhir sel dan cek sebagian cache secara manual.

In [ ]:
JA_RE = re.compile(r"[぀-ヿ一-鿿]")
def ja_ratio(s):
    # abaikan spasi, tanda baca (！？…、。), angka — kalimat tsundere penuh tanda baca dan
    # versi lama filter ini membuang banyak terjemahan yang sebenarnya bagus
    s = re.sub(r"[\s\W\d_]", "", s)
    return len(JA_RE.findall(s)) / max(1, len(s))

TRANSLATE_SYS = {
    "ja": ("You translate anime dialogue into natural, casual spoken Japanese. Keep the speaker's tone, "
           "emotion and stutters (e.g. 'I-I'm not...' -> 'わ、私は…'), and keep 'baka' as バカ. "
           "Output ONLY the Japanese line. No romaji, no quotes, no explanation."),
    "id": ("Translate the following chat message into natural, casual Indonesian, the way people text a friend "
           "(e.g. 'aku', 'kamu', 'nggak', 'banget'). Keep the meaning and emotion. "
           "Output ONLY the Indonesian message. No quotes, no explanation."),
}

def load_cache(lang):
    p = TRANSLATION_CACHE.format(lang=lang)
    return json.load(open(p, encoding="utf-8")) if os.path.exists(p) else {}
caches = {lang: load_cache(lang) for lang in ["ja", "id"]}

def translate_all(texts, lang, chunk=256):
    cache = caches[lang]
    todo = sorted({t for t in texts if t not in cache}, key=len)
    print(f"[{lang}] perlu diterjemahkan: {len(todo)} (cache: {len(cache)})")
    for i in range(0, len(todo), chunk):
        part = todo[i:i + chunk]
        res = generate([[{"role": "system", "content": TRANSLATE_SYS[lang]}, {"role": "user", "content": t}] for t in part],
                       max_new_tokens=200, greedy=True, bs=32)
        for src, out in zip(part, res):
            cache[src] = clean(out).strip('"「」')
        json.dump(cache, open(TRANSLATION_CACHE.format(lang=lang), "w", encoding="utf-8"), ensure_ascii=False)
        print(f"  {min(i + chunk, len(todo))}/{len(todo)}")

def tr(s, lang):
    """Terjemahan yang lolos cek kualitas, atau None."""
    if TARGET_LANG != "ja" or lang == "en":
        return s
    t = caches[lang].get(s, "")
    if not t or len(t) > 4 * len(s) + 40:          # kosong / ngelantur
        return None
    if lang == "ja":
        return t if ja_ratio(t) >= 0.6 else None
    return t if ja_ratio(t) < 0.1 else None         # "id": jangan sampai isinya huruf Jepang

if TARGET_LANG == "ja":
    translate_all([a for _, a in tsun_pairs] + [d for _, d in waifu_lines] + [u for u, _ in tsun_pairs], "ja")
    if "id" in USER_LANGS:
        translate_all([u for u, _ in tsun_pairs], "id")
    # laporan: berapa yang dibuang filter kualitas
    for name, items, lang in [("jawaban tsundere", [a for _, a in tsun_pairs], "ja"),
                              ("waifu lines", [d for _, d in waifu_lines], "ja"),
                              ("user → ja", [u for u, _ in tsun_pairs], "ja"),
                              ("user → id", [u for u, _ in tsun_pairs], "id")]:
        if lang in caches and (lang == "ja" or "id" in USER_LANGS):
            bad = [s for s in items if tr(s, lang) is None]
            print(f"{name}: dibuang {len(bad)}/{len(items)}", "| contoh:", [(s, caches[lang].get(s)) for s in bad[:2]])
    for lang in caches:
        for k in list(caches[lang])[:4]:
            print(f"[{lang}]", k, "->", caches[lang][k])

## 6. Susun train / eval / test

In [ ]:
def user_text(u):
    """Pilih bahasa user sesuai porsi USER_LANGS; kalau terjemahannya gagal, coba bahasa lain."""
    if TARGET_LANG != "ja":
        return u
    langs = random.choices(list(USER_LANGS), weights=list(USER_LANGS.values()))[:1] + list(USER_LANGS)
    for lang in langs:
        t = tr(u, lang)
        if t:
            return t
    return None

tsun_conv = []
for u, a in tsun_pairs:
    uu, aa = user_text(u), tr(a, "ja")
    if uu and aa:
        tsun_conv.append([{"role": "system", "content": SYSTEM_PROMPT},
                          {"role": "user", "content": uu},
                          {"role": "assistant", "content": aa}])

test_conv  = tsun_conv[:TEST_SIZE]       # ditahan, tidak ikut training
train_tsun = tsun_conv[TEST_SIZE:]

waifu_conv = []
for trait, d in waifu_lines:
    aa = tr(d, "ja")
    if aa:
        waifu_conv.append([{"role": "system", "content": persona_prompt(trait)},
                           {"role": "user", "content": generic_prompt()},
                           {"role": "assistant", "content": aa}])

all_train = train_tsun + waifu_conv + synth
random.shuffle(all_train)

from datasets import Dataset
EMPTY_THINK = "<think>\n\n</think>\n\n"
def to_text(msgs):
    # Template Qwen3 menyisipkan blok <think></think> kosong di jawaban terakhir.
    # Model Instruct-2507 tidak memakai thinking, jadi blok ini dibuang; kalau tidak,
    # model hasil fine-tune akan mengawali setiap jawaban dengan tag sampah (<tool_call>).
    t = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    return t.replace(EMPTY_THINK, "")

ds = Dataset.from_dict({"text": [to_text(m) for m in all_train]})
ds = ds.filter(lambda x: len(tokenizer(x["text"], add_special_tokens=False)["input_ids"]) <= MAX_SEQ_LEN)
split = ds.train_test_split(test_size=0.03, seed=SEED)
train_ds, eval_ds = split["train"], split["test"]
print(f"train: {len(train_ds)} | eval(loss): {len(eval_ds)} | test(generate): {len(test_conv)}")
print("tsundere:", len(train_tsun), "| waifu:", len(waifu_conv), "| synth:", len(synth))
assert not any("<think>" in t for t in train_ds["text"][:200]), "Masih ada blok <think> di data training!"
print(train_ds[0]["text"][:800])

## 7. Baseline (sebelum training)
Skor sederhana: penanda tsundere (stutter, バカ/baka, 別に/"it's not like", ふん/hmph, …) dan, untuk mode `ja`, rasio karakter Jepang. Ini hanya heuristik kasar. Baca juga contoh jawabannya dan nilai sendiri.

In [ ]:
MARKERS = {
    "ja": [r"(.)、\1", r"バカ|ばか", r"別に", r"ふん|フン", r"勘違い", r"じゃないんだから", r"…|・・・"],
    "en": [r"\b(\w)-\1", r"\bbaka\b|\bidiot\b|\bdummy\b", r"it'?s not like", r"\bhmph\b", r"don'?t get the wrong idea", r"\.\.\.|…"],
}[TARGET_LANG]

MARKERS.append(r"うるさい")

def score(raw_replies):
    n = len(raw_replies)
    tags = sum(bool(TAG_RE.search(r)) for r in raw_replies) / n
    replies = [clean(r) for r in raw_replies]
    tsun = sum(any(re.search(p, r, re.I) for p in MARKERS) for r in replies) / n
    lang = sum(ja_ratio(r) >= 0.8 for r in replies) / n if TARGET_LANG == "ja" else None
    empty = sum(not r for r in replies) / n
    return {"tsundere": round(tsun, 2), "japanese": lang and round(lang, 2),
            "empty": round(empty, 2), "tag_leak": round(tags, 2), "avg_chars": round(sum(map(len, replies)) / n)}

# Tes ala aplikasi: pesan user campur Indonesia/Jepang + pertanyaan fakta (jawaban yang benar harus muncul)
APP_TESTS = [("Halo Sakura sayang, kamu cantik banget hari ini.", None), ("Sakura, aku capek banget habis kerja.", None),
             ("Kamu lagi ngapain?", None), ("Aku bawain kue buat kamu nih.", None), ("Sakura, berapa 15 kali 4?", "60"),
             ("Ibu kota Jepang apa?", "東京"), ("おはよう、サクラ！", None), ("サクラのこと好きだよ。", None),
             ("1から10まで足すといくつ？", "55"), ("Aku mau pergi main sama temen cewek ya.", None)] * 2
app_prompts = [[{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": q}] for q, _ in APP_TESTS]

def app_score(raw):
    s = score(raw)
    facts = [(a in clean(r)) for (q, a), r in zip(APP_TESTS, raw) if a]
    s["facts"] = f"{sum(facts)}/{len(facts)}"
    return s

test_prompts = [c[:2] for c in test_conv]
base_replies = generate(test_prompts)
base_app = generate(app_prompts)
print("BASELINE held-out:", score(base_replies))
print("BASELINE app-test:", app_score(base_app))
for c, r in list(zip(test_conv, base_replies))[:5]:
    print("\nUSER:", c[1]["content"], "\nBASE:", clean(r), "\nREF :", c[2]["content"])

## 8. Training (SFT, loss hanya di jawaban assistant)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

FastLanguageModel.for_training(model)
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        max_steps=MAX_STEPS,
        learning_rate=LR,
        warmup_ratio=0.03,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        weight_decay=0.01,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        eval_strategy="steps", eval_steps=200,
        save_strategy="steps", save_steps=200, save_total_limit=2,
        output_dir=OUTPUT_DIR,
        seed=SEED,
        report_to="none",
        dataset_num_proc=2,
    ),
)
trainer = train_on_responses_only(trainer,
    instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")

# Resume otomatis kalau ada checkpoint di Drive (misal Colab terputus)
has_ckpt = bool(glob.glob(f"{OUTPUT_DIR}/checkpoint-*"))
stats = trainer.train(resume_from_checkpoint=has_ckpt or None)
print(stats)

## 9. Evaluasi setelah training
**Ekspor ke GGUF hanya kalau FINETUNED di app-test minimal setara BASELINE**: `japanese` dan `facts` tidak turun, `empty` dan `tag_leak` = 0, `tsundere` naik. Model dasar + system prompt sudah cukup kuat (di laptop: japanese 20/20, facts 6/6, tsundere 15/20). Kalau fine-tune kalah, lebih baik tetap pakai model dasar.

In [ ]:
ft_replies = generate(test_prompts)
ft_app = generate(app_prompts)
print("held-out  BASELINE :", score(base_replies))
print("held-out  FINETUNED:", score(ft_replies))
print("app-test  BASELINE :", app_score(base_app))
print("app-test  FINETUNED:", app_score(ft_app))
for (q, _), b, f in list(zip(APP_TESTS, base_app, ft_app))[:10]:
    print("\nUSER:", q, "\nBASE:", clean(b), "\nFT  :", f)     # FT ditampilkan mentah supaya tag bocor terlihat

## 10. Simpan adapter LoRA + ekspor GGUF (Q4_K_M) ke Drive

In [ ]:
model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)
print("LoRA disimpan:", LORA_DIR)

In [ ]:
# Gabung LoRA ke model lalu kuantisasi ke GGUF (Unsloth membangun llama.cpp otomatis, ±10–20 menit)
model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method="q4_k_m")

import shutil
ggufs = glob.glob(f"{GGUF_DIR}/**/*.gguf", recursive=True) + glob.glob("/content/*.gguf")
for g in ggufs:
    dst = f"{DRIVE_ROOT}/sakura-qwen3-4b-{TARGET_LANG}-{RUN_NAME}-Q4_K_M.gguf" if "q4_k_m" in g.lower() else f"{DRIVE_ROOT}/{os.path.basename(g)}"
    shutil.copy(g, dst)
    print("Disalin ke Drive:", dst, f"({os.path.getsize(dst)/1e9:.2f} GB)")

## Pakai di laptop (LM Studio)
1. Unduh **file** `sakura-qwen3-4b-ja-v2-Q4_K_M.gguf` (±2.5 GB) dari Drive — klik kanan file-nya → Download, jangan unduh satu folder (Drive akan memecahnya jadi banyak zip).
2. Taruh di folder model LM Studio, misalnya `<models>/local/sakura-qwen3-4b/sakura-qwen3-4b-ja-Q4_K_M.gguf`, lalu load.
3. Ubah `.env`: `LM_STUDIO_MODEL=<nama model seperti yang tampil di LM Studio>`.
4. `source/system_prompt.txt` di aplikasi harus sama dengan `SYSTEM_PROMPT` di atas.

Atribusi: dataset *anime-waifu-personality-chat* oleh scryptiam (CC-BY-4.0).